# Sales at five price points: Lorenz curve

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Lorenz Curve and Gini Coefficient** · a worked example from the StatNex teaching guide

**The story.** An online shop sells products at five price points, from \$10 to \$400, and counts the buyers at each one.

**The question.** Do a few expensive sales bring in a large share of revenue?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/lorenz_curve__prices.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Product price ($),Number of buyers
10.0,620.0
25.0,410.0
50.0,240.0
100.0,95.0
400.0,30.0

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
# Grouped data: one row per class, its average value in one column and, in each
# frequency column, how many units fall in it. A class with no units is skipped.
class_name = "Product price ($)"
means = pd.to_numeric(data[class_name], errors="coerce")
inputs = {}
for name in ["Number of buyers"]:
    weights = pd.to_numeric(data[name], errors="coerce")
    usable = means.notna() & weights.notna() & (weights > 0)
    inputs[name] = (means[usable].to_numpy(), weights[usable].to_numpy())

## Lorenz curve and concentration indices

In [ ]:
# Units sorted from smallest to largest. The curve plots the cumulative share of the
# population (p) against the cumulative share of the total they hold (L). SciPy and
# statsmodels have no Lorenz or Gini function, so both are written out.
#   Gini    = twice the area between the diagonal and the curve (trapezium rule,
#             exact because the curve is made of straight pieces): 0 when everyone
#             holds the same, near 1 when one unit holds everything. This is the
#             plain Gini, without the n/(n - 1) "unbiased" factor some packages add.
#   HHI     = sum of squared shares; 1/n when everything is equal
#   Entropy = -sum(share * log(share)); log(n) when everything is equal
#   Pietra  = the widest vertical gap between the diagonal and the curve: the share
#             of the total that would have to move to make everyone equal
# With grouped data the HHI and the entropy take each class's total as one share
# (how the total splits across the classes), while the curve and the Gini count
# every unit. A curve built from class or decile averages ignores the spread inside
# each class, so its Gini understates the Gini across the individual units.


def lorenz(values, weights):
    order = np.argsort(values, kind="stable")
    values, weights = values[order], weights[order]
    amount = (values * weights).sum()
    p = np.concatenate([[0.0], np.cumsum(weights) / weights.sum()])
    curve = np.concatenate([[0.0], np.cumsum(values * weights) / amount])
    share = values * weights / amount
    positive = share[share > 0]
    return {
        "p": p, "curve": curve, "n": len(values),
        "gini": 1 - ((p[1:] - p[:-1]) * (curve[1:] + curve[:-1])).sum(),
        "hhi": (share**2).sum(),
        "entropy": -(positive * np.log(positive)).sum(),
        "pietra": (p - curve).max(),
    }


results = {name: lorenz(*pair) for name, pair in inputs.items()}
indices = pd.DataFrame({name: [r["gini"], r["hhi"], r["hhi"] * 10000, r["entropy"]]
                        for name, r in results.items()},
                       index=["Gini coefficient", "HHI (ratio)", "HHI (%)", "Entropy"])
print(indices)
extra = pd.DataFrame({name: [r["pietra"], 1 / r["n"], np.log(r["n"])]
                      for name, r in results.items()},
                     index=["Pietra index", "HHI lower bound (1/n)", "Entropy upper bound (log n)"])
print()
print(extra)

## Points of the curve

In [ ]:
# Every series on one grid of p, interpolating where a series has no point of its own.
grid = sorted({value for r in results.values() for value in r["p"]})
chart = pd.DataFrame({"p": grid, **{name: np.interp(grid, r["p"], r["curve"])
                                    for name, r in results.items()}})
print()
print(chart.to_string(index=False))

## Lorenz curve

In [ ]:
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Perfect equality")
for name, r in results.items():
    ax.plot(r["p"], r["curve"], marker="o", label=name)
ax.set_xlabel("Cumulative share of the population")
ax.set_ylabel("Cumulative share of the total")
ax.legend()
fig.savefig("figures/lorenz.png", dpi=120, bbox_inches="tight")

## What the result says

Here the Lorenz curve puts the purchases in order from the cheapest product to the most expensive and shows the share of revenue they bring in together. The 620 buyers of the \$10 product are 0.444 of all buyers, about 44%, but they bring in only 0.124 of revenue, about 12%. All buyers except the 30 who bought the \$400 product bring in 0.76 of revenue. So those 30 buyers, about 2% of the total, bring in nearly a quarter.

The Gini coefficient measures concentration from 0 (every sale brings the same) to 1 (one sale brings everything). Here it is 0.53.

So yes, a few expensive sales bring in a large share of revenue. But the curve counts purchases, not people, and it shows only what each sale brings in by itself.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.